# Indicator Discovery Lab

Отдельный исследовательский стенд для ответа на один вопрос: **какие causal-индикаторы, сочетания и компактные модели действительно переносятся на следующий временной блок для разных targets, валют и горизонтов?**

Здесь нет meta-model и клиентской policy. Параметры каждого правила выбираются на rolling train, после чего фиксируются на следующем полугодовом OOS-fold. Все итоговые метрики объединяют только OOS-предсказания. Период после `discovery_end` зарезервирован и не участвует ни в выборе, ни в таблицах.

In [ ]:
from datetime import date
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((
    path.resolve() for path in candidate_roots
    if (path / 'src' / 'cbr_loader.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден корень prod с папкой src')
project_root_str = str(PROJECT_ROOT)
sys.path = [item for item in sys.path if item != project_root_str]
sys.path.insert(0, project_root_str)
cached_src = sys.modules.get('src')
expected_src = (PROJECT_ROOT / 'src').resolve()
if cached_src is not None:
    cached_file = Path(getattr(cached_src, '__file__', '') or '.').resolve()
    if expected_src not in cached_file.parents:
        for module_name in [name for name in tuple(sys.modules) if name == 'src' or name.startswith('src.')]:
            del sys.modules[module_name]

from src.cbr_loader import CURRENCIES, load_cbr_history
from src.features import build_features
from src.market_data import build_daily_market_panel
from src.outcomes import add_future_outcomes
from research.yura.src.advanced_indicators import (
    ADVANCED_FEATURES, add_advanced_indicator_features,
    discovery_indicator_rules,
)
from research.yura.src.config import YuraPipelineConfig
from research.yura.src.discovery_targets import build_discovery_targets
from research.yura.src.indicator_discovery import (
    IndicatorDiscoveryConfig, build_discovery_temporal_plan,
    run_rule_indicator_discovery,
    summarize_discovery_folds,
)
from research.yura.src.model_indicator_discovery import (
    MODEL_INDICATOR_SPECS, run_model_indicator_discovery,
)

START_DATE = date(2018, 1, 1)
END_DATE = date.today()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cbr'
CURRENCIES_TO_TEST = YuraPipelineConfig().currencies
HORIZONS = YuraPipelineConfig().horizons
DISCOVERY_CONFIG = IndicatorDiscoveryConfig(
    train_months=36, test_months=6, discovery_oos_months=48,
    min_train_signals_per_week=0.25, max_train_signals_per_week=4.0,
    top_rules_per_family=2, max_pair_families=12,
    precision_prior_strength=20.0, min_oos_signals=20,
)
# Полный набор сравнивает линейную модель, HGB до/после advanced features
# и Extra Trees. Для короткого smoke-run можно оставить ('hgb_core', 'hgb_all').
MODEL_SPECS = MODEL_INDICATOR_SPECS
DISCOVERY_CONFIG

## 1. Данные, causal features и четыре target-концепции

Используются четыре исходные концепции — `G0`, `G1`, `W0`, `W1`. У `G1` остаются три заранее заданных tolerance-варианта: это разные определения одного target-концепта, и результаты показываются отдельно.

In [ ]:
cbr_history = load_cbr_history(
    start_date=START_DATE, end_date=END_DATE, currencies=CURRENCIES, raw_dir=RAW_DIR,
)
rates = (
    cbr_history.pivot(index='available_at', columns='currency', values='normalized_rate')
    .reindex(columns=list(CURRENCIES)).sort_index()
)
rates.columns.name = None
market_panel = build_daily_market_panel(rates)
features = build_features(market_panel)
features = add_advanced_indicator_features(features)
outcomes = add_future_outcomes(features, horizons=HORIZONS)
dataset, discovery_target_registry = build_discovery_targets(
    outcomes, horizons=HORIZONS,
)
discovery_data = dataset.loc[
    dataset['is_update_day'] & dataset['currency'].isin(CURRENCIES_TO_TEST)
].copy()
discovery_temporal_plan = build_discovery_temporal_plan(
    discovery_data, DISCOVERY_CONFIG
)
DISCOVERY_END = pd.Timestamp(discovery_temporal_plan.discovery_end.iloc[0])

display(discovery_temporal_plan)
display(discovery_target_registry[[
    'family', 'name', 'scenario', 'horizon', 'description'
]])
pd.DataFrame({
    'first_available_at': [discovery_data['available_at'].min()],
    'last_available_at': [discovery_data['available_at'].max()],
    'rows': [len(discovery_data)],
    'advanced_features': [len(ADVANCED_FEATURES)],
})

In [ ]:
target_prevalence = []
for definition in discovery_target_registry.itertuples(index=False):
    eligible = (
        discovery_data['available_at'].lt(DISCOVERY_END)
        & discovery_data['available_at'].add(pd.Timedelta(days=int(definition.horizon))).lt(DISCOVERY_END)
    )
    values = discovery_data.loc[eligible, definition.name].dropna().astype(int)
    target_prevalence.append({
        'target_family': definition.family, 'target': definition.name,
        'horizon': definition.horizon, 'observations': len(values),
        'positive_count': int(values.sum()), 'positive_rate': float(values.mean()),
    })
target_prevalence = pd.DataFrame(target_prevalence)
display(target_prevalence)

## 2. Каталог rule-гипотез

Включены старые level/momentum/streak/trend/volatility/calendar/context признаки и удалённые из основного pipeline causal Kalman, directional CUSUM, mean-shift, volatility-shift и rare-low-reversal. Конкретные window/threshold не считаются отдельными победителями: внутри каждого fold выбирается лучшая параметризация семейства.

In [ ]:
RULES = discovery_indicator_rules()
rule_catalogue = pd.DataFrame([{
    'family': rule.family, 'feature': rule.feature,
    'operator': rule.operator, 'threshold': rule.threshold,
    'advanced': rule.feature in ADVANCED_FEATURES,
} for rule in RULES])
display(rule_catalogue.groupby(['family', 'advanced']).agg(
    variants=('feature', 'size'), features=('feature', 'nunique')
).reset_index())
print(f'Concrete rules: {len(RULES):,}; hypothesis families: {rule_catalogue.family.nunique():,}')

## 3. Walk-forward простых правил и AND/OR

Каждая строка итогового leaderboard — не один удачно найденный threshold, а переобучаемый engine: на каждом rolling train он заново выбирает concrete rule, затем выдаёт сигналы на следующем OOS-fold. Pair search предварительно оставляет 12 лучших train-семейств и не читает test.

In [ ]:
rule_result = run_rule_indicator_discovery(
    discovery_data, target_registry=discovery_target_registry,
    rules=RULES, currencies=CURRENCIES_TO_TEST, config=DISCOVERY_CONFIG,
)
display(rule_result.temporal_plan)
print(f'Rule fold records: {len(rule_result.fold_results):,}')
display(rule_result.best_by_configuration[[
    'currency', 'target_family', 'target', 'horizon',
    'strategy_kind', 'strategy_name', 'logic', 'quality_group',
    'oos_signal_count', 'oos_precision', 'oos_lift', 'oos_lift_lcb95',
    'oos_benefit_uplift_bps', 'fold_share_lift_gt_1',
]])

In [ ]:
rule_quality_distribution = (
    rule_result.leaderboard.groupby(
        ['target_family', 'strategy_kind', 'quality_group'], sort=True
    ).size().rename('configurations').reset_index()
)
display(rule_quality_distribution)
display(rule_result.family_summary.groupby('target_family', group_keys=False).head(15))

## 4. Компактные модели как самостоятельные индикаторы

Для каждого target-варианта и горизонта модель pooled по валютам. Внутри каждого 36-месячного train соблюдены три последовательных блока: `24m estimator fit → 6m probability calibration → 6m threshold validation`; затем идёт отдельный `6m OOS test`. Сравнение `hgb_core` и `hgb_all` напрямую показывает добавочную ценность advanced features.

In [ ]:
model_result = run_model_indicator_discovery(
    discovery_data, target_registry=discovery_target_registry,
    currencies=CURRENCIES_TO_TEST, model_specs=MODEL_SPECS,
    config=DISCOVERY_CONFIG,
)
display(model_result.catalogue)
print(f'Model fold records: {len(model_result.fold_results):,}')
display(model_result.best_by_configuration[[
    'currency', 'target_family', 'target', 'horizon', 'strategy_name',
    'quality_group', 'oos_signal_count', 'oos_precision', 'oos_lift',
    'oos_lift_lcb95', 'oos_benefit_uplift_bps', 'fold_share_lift_gt_1',
]])

## 5. Единый leaderboard и победители конфигураций

`oos_lift` — точечная оценка; `oos_lift_lcb95` — консервативная нижняя граница Wilson для precision, делённая на частоту target. Ранг строится по LCB, поэтому правило `2 из 2` не обгоняет поддержанный сигнал только из-за precision=100%. `quality_group` — прозрачная диагностическая маркировка, не часть обучения.

In [ ]:
all_discovery_folds = pd.concat([
    rule_result.fold_results, model_result.fold_results
], ignore_index=True)
indicator_leaderboard, best_indicators, indicator_family_summary = summarize_discovery_folds(
    all_discovery_folds, min_oos_signals=DISCOVERY_CONFIG.min_oos_signals,
)
top_indicators = (
    indicator_leaderboard.groupby(
        ['currency', 'target_family', 'target', 'horizon'], group_keys=False
    ).head(10).reset_index(drop=True)
)
display(best_indicators[[
    'currency', 'target_family', 'target', 'horizon', 'strategy_kind',
    'strategy_name', 'logic', 'quality_group', 'oos_signal_count',
    'oos_precision', 'oos_random_precision', 'oos_lift', 'oos_lift_lcb95',
    'oos_benefit_uplift_bps', 'fold_lift_p25', 'fold_share_lift_gt_1',
    'parameter_stability',
]])
display(top_indicators.head(100))

In [ ]:
winner_counts = (
    best_indicators.groupby(
        ['target_family', 'strategy_kind', 'strategy_name', 'logic'], sort=True
    ).agg(
        wins=('target', 'size'),
        strong_wins=('quality_group', lambda values: int(values.eq('strong').sum())),
        median_lift=('oos_lift', 'median'),
        median_lift_lcb95=('oos_lift_lcb95', 'median'),
        median_bps=('oos_benefit_uplift_bps', 'median'),
    ).reset_index().sort_values(
        ['target_family', 'strong_wins', 'wins', 'median_lift_lcb95'],
        ascending=[True, False, False, False],
    )
)
display(winner_counts.groupby('target_family', group_keys=False).head(20))

## 6. Проверка ценности комбинаций и advanced features

In [ ]:
configuration_keys = ['currency', 'target_family', 'target', 'horizon']
best_single = (
    indicator_leaderboard.loc[indicator_leaderboard.strategy_kind.eq('single')]
    .sort_values([*configuration_keys, 'oos_lift_lcb95'], ascending=[True]*4 + [False])
    .drop_duplicates(configuration_keys)
    [configuration_keys + ['oos_lift_lcb95', 'oos_benefit_uplift_bps']]
    .rename(columns={
        'oos_lift_lcb95': 'single_lift_lcb95',
        'oos_benefit_uplift_bps': 'single_bps',
    })
)
best_combo = (
    indicator_leaderboard.loc[indicator_leaderboard.strategy_kind.eq('combination')]
    .sort_values([*configuration_keys, 'oos_lift_lcb95'], ascending=[True]*4 + [False])
    .drop_duplicates(configuration_keys)
    [configuration_keys + ['strategy_name', 'logic', 'oos_lift_lcb95', 'oos_benefit_uplift_bps']]
)
combination_increment = best_combo.merge(best_single, on=configuration_keys, how='inner')
combination_increment['delta_lift_lcb95'] = combination_increment.oos_lift_lcb95 - combination_increment.single_lift_lcb95
combination_increment['delta_bps'] = combination_increment.oos_benefit_uplift_bps - combination_increment.single_bps
display(combination_increment.sort_values('delta_lift_lcb95', ascending=False))

In [ ]:
hgb_comparison = indicator_leaderboard.loc[
    indicator_leaderboard.strategy_name.isin(['hgb_core', 'hgb_all'])
].pivot_table(
    index=configuration_keys, columns='strategy_name',
    values=['oos_lift_lcb95', 'oos_benefit_uplift_bps'], aggfunc='first',
)
if not hgb_comparison.empty:
    hgb_comparison['delta_lift_lcb95_all_minus_core'] = (
        hgb_comparison[('oos_lift_lcb95', 'hgb_all')]
        - hgb_comparison[('oos_lift_lcb95', 'hgb_core')]
    )
    hgb_comparison['delta_bps_all_minus_core'] = (
        hgb_comparison[('oos_benefit_uplift_bps', 'hgb_all')]
        - hgb_comparison[('oos_benefit_uplift_bps', 'hgb_core')]
    )
display(hgb_comparison.reset_index())

## 7. Карты работоспособности

Цвет показывает медиану консервативного OOS lift лучших десяти методов конфигурации. Это карта областей, где текущая библиотека вообще содержит устойчивое evidence, а не финальный backtest продукта.

In [ ]:
plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F4F4F4',
    'axes.edgecolor': '#222222', 'text.color': '#111111',
    'axes.labelcolor': '#111111', 'xtick.color': '#111111',
    'ytick.color': '#111111',
})
families = list(discovery_target_registry.family.drop_duplicates())
fig, axes = plt.subplots(len(families), 1, figsize=(11, 4 * len(families)), constrained_layout=True)
for ax, family in zip(np.atleast_1d(axes), families):
    sample = top_indicators.loc[top_indicators.target_family.eq(family)]
    matrix = sample.pivot_table(
        index='currency', columns='horizon', values='oos_lift_lcb95', aggfunc='median'
    )
    matrix = matrix.reindex(index=CURRENCIES_TO_TEST, columns=HORIZONS)
    values = matrix.to_numpy(dtype=float)
    image = ax.imshow(values, cmap='Greys', aspect='auto', vmin=0, vmax=max(1.3, np.nanpercentile(values, 95)))
    ax.set_xticks(range(len(HORIZONS)), labels=[f'h={h}' for h in HORIZONS])
    ax.set_yticks(range(len(CURRENCIES_TO_TEST)), labels=CURRENCIES_TO_TEST)
    ax.set_title(f'{family}: median top-10 OOS lift LCB95', fontweight='bold')
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            ax.text(column, row, '—' if not np.isfinite(value) else f'{value:.2f}',
                    ha='center', va='center', color='black', fontweight='bold')
    fig.colorbar(image, ax=ax, label='OOS lift LCB95')
plt.show()

## 8. Как использовать результаты

1. Начинать с `best_indicators` и `winner_counts`, но проверять `oos_signal_count`, `oos_lift_lcb95`, BPS и стабильность fold одновременно.
2. Новый метод имеет смысл переносить в Yura, если он выигрывает не одну строку, а повторяется по нескольким валютам/горизонтам или улучшает HGB в сравнении `hgb_all − hgb_core`.
3. Высокий точечный lift при низком LCB или одном удачном fold — исследовательский шум.
4. `parameter_stability` показывает, насколько часто rolling fit выбирал одну и ту же конкретную настройку. Низкое значение не всегда плохо, но означает реальную необходимость переоптимизации.
5. Период `reserved_after` не использован. После выбора небольшой новой библиотеки её нужно один раз заморозить и проверить уже полным Yura pipeline.